# spaCy Statistical NER Training (Phase 4)

**Purpose**: Train statistical NER model using distant supervision data

**Input**: `.spacy` files from Phase 3 (train/dev/test)

**Output**: Trained spaCy model with generalization capability

**Runtime**: ~10-30 minutes on Google Colab GPU (T4/V100/A100)

---

## Instructions

1. **Runtime**: Select GPU runtime (Runtime → Change runtime type → GPU)
2. **Execute cells**: Run cells in order from top to bottom
3. **Monitor training**: Watch F1 scores in training output
4. **Save model**: Final cell copies model back to Google Drive

---

## Cell 1: Setup Environment

In [ ]:
# Install spaCy
!pip install -U spacy==3.7.0

# Verify GPU is available
!nvidia-smi

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set project paths
PROJECT_ROOT = '/content/drive/MyDrive/inventory_2022'
SPACY_ROOT = f'{PROJECT_ROOT}/spacy_hybrid_ner'

print(f"\nProject root: {PROJECT_ROOT}")
print(f"spaCy root: {SPACY_ROOT}")

## Cell 3: Copy Training Data to Colab

Copy .spacy files from Drive to Colab local storage for faster training.

In [ ]:
import os

# Create directories
!mkdir -p ./data/ner_training
!mkdir -p ./models

# Copy files from Drive to Colab
!cp {SPACY_ROOT}/data/ner_training/train.spacy ./data/ner_training/
!cp {SPACY_ROOT}/data/ner_training/dev.spacy ./data/ner_training/
!cp {SPACY_ROOT}/data/ner_training/test.spacy ./data/ner_training/
!cp {SPACY_ROOT}/data/ner_training/config.cfg ./

print("\nFiles copied successfully!")
!ls -lh ./data/ner_training/

## Cell 4: Verify Training Data

In [ ]:
import spacy
from spacy.tokens import DocBin

# Initialize blank spaCy
nlp = spacy.blank("en")

# Load training data
db_train = DocBin().from_disk('./data/ner_training/train.spacy')
docs_train = list(db_train.get_docs(nlp.vocab))

db_dev = DocBin().from_disk('./data/ner_training/dev.spacy')
docs_dev = list(db_dev.get_docs(nlp.vocab))

db_test = DocBin().from_disk('./data/ner_training/test.spacy')
docs_test = list(db_test.get_docs(nlp.vocab))

print(f"Train documents: {len(docs_train)}")
print(f"Dev documents: {len(docs_dev)}")
print(f"Test documents: {len(docs_test)}")

# Sample entities from training data
print("\nSample entities from training data:")
for i, doc in enumerate(docs_train[:3]):
    print(f"\nDoc {i+1}:")
    print(f"  Text: {doc.text[:80]}...")
    print(f"  Entities ({len(doc.ents)}):")
    for ent in doc.ents[:5]:
        print(f"    - '{ent.text}' ({ent.label_})")

## Cell 5: Train Model

Train statistical NER model using spaCy's training CLI.

**Expected Duration**: 10-30 minutes depending on GPU type

**Target Metrics**: F1 > 70% on dev set

In [ ]:
%%bash
# Train with GPU
python -m spacy train \
  config.cfg \
  --output ./models/ner_statistical \
  --paths.train ./data/ner_training/train.spacy \
  --paths.dev ./data/ner_training/dev.spacy \
  --gpu-id 0 \
  --verbose

## Cell 6: Test on Sample Text

In [ ]:
import spacy

# Load best model
nlp = spacy.load("./models/ner_statistical/model-best")

# Test on sample text
test_text = """
The Clinical Genome Resource (ClinGen) and the OMIM database provide
comprehensive genetic information. We also analyzed data from the
European Nucleotide Archive (ENA) and compared it with UniProt entries.
A new Genomics Repository was created for this study.
"""

doc = nlp(test_text)

print("Extracted Entities:")
for ent in doc.ents:
    print(f"  - '{ent.text}' ({ent.label_})")

## Cell 7: Evaluate on Test Set

In [ ]:
%%bash
# Quantitative evaluation on test set
python -m spacy evaluate \
  ./models/ner_statistical/model-best \
  ./data/ner_training/test.spacy \
  --output ./models/ner_statistical/test_evaluation.json \
  --gpu-id 0

## Cell 8: Analyze Evaluation Results

In [ ]:
import json

# Load evaluation results
with open('./models/ner_statistical/test_evaluation.json', 'r') as f:
    results = json.load(f)

print("\n" + "=" * 60)
print("TEST SET EVALUATION RESULTS")
print("=" * 60)
print(f"\nOverall NER Performance:")
print(f"  Precision: {results['ents_p']:.4f}")
print(f"  Recall:    {results['ents_r']:.4f}")
print(f"  F1 Score:  {results['ents_f']:.4f}")

if 'ents_per_type' in results:
    print(f"\nPer-Type Performance:")
    for label, metrics in results['ents_per_type'].items():
        print(f"\n  {label}:")
        print(f"    Precision: {metrics['p']:.4f}")
        print(f"    Recall:    {metrics['r']:.4f}")
        print(f"    F1:        {metrics['f']:.4f}")

print("\n" + "=" * 60)

# Success check
target_f1 = 0.70
if results['ents_f'] >= target_f1:
    print(f"\n✓ SUCCESS: F1 score {results['ents_f']:.4f} exceeds target {target_f1}")
else:
    print(f"\n⚠️  WARNING: F1 score {results['ents_f']:.4f} below target {target_f1}")
    print("   Consider: (1) Train longer, (2) Tune hyperparameters, (3) Improve annotation quality")

## Cell 9: Test Generalization to NEW Entities

Test if model can discover NEW bioresources not in training dictionary.

In [ ]:
# Test generalization on NEW entities (not in dictionary)
test_cases = [
    "We created the Genomics Knowledge Base (GKB) for this study.",  # NEW!
    "The Cell Atlas Repository contains single-cell data.",          # NEW!
    "Data from PDB and UniProt were integrated.",                     # KNOWN
    "The Proteomics Data Commons (PDC) was recently launched.",       # NEW!
    "We used MGI and FlyBase for comparative analysis.",              # KNOWN
]

nlp = spacy.load("./models/ner_statistical/model-best")

print("Testing generalization to NEW entities...")
print("=" * 70)

for i, text in enumerate(test_cases, 1):
    doc = nlp(text)
    print(f"\nTest {i}:")
    print(f"  Text: {text}")

    entities = [(ent.text, ent.label_) for ent in doc.ents]
    if entities:
        print(f"  Entities: {entities}")
        status = "NEW" if "NEW" in text else "KNOWN"
        print(f"  Status: ✓ {status} entity detected")
    else:
        print(f"  Entities: []")
        print(f"  Status: ✗ No entities found")

print("\n" + "=" * 70)
print("If NEW entities are detected, statistical model is generalizing correctly!")

## Cell 10: Save Model to Google Drive

In [ ]:
# Copy trained model back to Drive
!cp -r ./models/ner_statistical/model-best {SPACY_ROOT}/models/ner_statistical

print("\n" + "=" * 70)
print("MODEL SAVED TO GOOGLE DRIVE")
print("=" * 70)
print(f"Location: {SPACY_ROOT}/models/ner_statistical")
print("\nNext steps:")
print("  1. Download model for local validation")
print("  2. Build hybrid pipeline (EntityRuler + Statistical NER)")
print("  3. Evaluate hybrid pipeline on test set")
print("=" * 70)